# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)


True

In [3]:
client = OpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    base_url="https://cx-test-foundry.openai.azure.com/openai/v1/",
)

In [4]:

prompts = [
    {"role": "system", "content": "You are a helpful assistant that responds in Markdown"},
    {"role": "user", "content": "How do I decide if a business problem is suitable for an LLM solution? Please respond in Markdown."}
  ] 
MODEL="grok-4-1-fast-reasoning"
# Connect to OpenAI
def getClient(model="grok-4-1-fast-reasoning",messages=prompts):

    completion = client.chat.completions.create(
    model=model, # Replace with your model deployment name.
    messages=prompts
    )
    return completion

In [5]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [6]:
ed = Website("https://www.seic.com/")
ed.links

['https://www.seic.com/',
 'https://www.seic.com/',
 '/about-sei/overview',
 'https://www.seic.com/home-page?nohpredirect=',
 'https://www.seic.com/en-gb?nohpredirect=',
 'https://www.seic.com/en-ca?nohpredirect=',
 'https://www.seic.com/fr-ca?nohpredirect=',
 'https://www.seic.com/home-page?nohpredirect=',
 'https://www.seic.com/en-gb?nohpredirect=',
 'https://www.seic.com/en-ca?nohpredirect=',
 'https://www.seic.com/fr-ca?nohpredirect=',
 'https://www.seic.com/asset-managers/overview',
 'https://www.seic.com/banks-and-wealth-managers/overview',
 'https://www.seic.com/financial-advisors/overview',
 'https://www.seic.com/institutional-investors/overview',
 '/about-sei/about-sei',
 '/ent/client-site-logins-and-contact-information',
 '/about-sei/contact-us',
 '/about-sei/locations',
 '/about-sei/newsroom',
 'https://ir.seic.com/',
 'https://careers.seic.com/',
 'https://www.facebook.com/SEICorporateHeadquarters/',
 'https://www.instagram.com/sei_hq',
 'https://www.linkedin.com/company/se

## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [7]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}
"""

In [8]:
print(link_system_prompt)

You are provided with a list of links found on a webpage. You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}



In [9]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [10]:
print(get_links_user_prompt(ed))

Here is the list of links on the website of https://www.seic.com/ - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
https://www.seic.com/
https://www.seic.com/
/about-sei/overview
https://www.seic.com/home-page?nohpredirect=
https://www.seic.com/en-gb?nohpredirect=
https://www.seic.com/en-ca?nohpredirect=
https://www.seic.com/fr-ca?nohpredirect=
https://www.seic.com/home-page?nohpredirect=
https://www.seic.com/en-gb?nohpredirect=
https://www.seic.com/en-ca?nohpredirect=
https://www.seic.com/fr-ca?nohpredirect=
https://www.seic.com/asset-managers/overview
https://www.seic.com/banks-and-wealth-managers/overview
https://www.seic.com/financial-advisors/overview
https://www.seic.com/institutional-investors/overview
/about-sei/about-sei
/ent/client-site-logins-and-contact-information
/about-sei/contact-us
/about-sei

In [11]:
def get_links(url):
    website = Website(url)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
    response_format={"type": "json_object"}
    )
    result= response.choices[0].message.content
    return json.loads(result)

In [12]:
# Anthropic has made their site harder to scrape, so I'm using HuggingFace..

huggingface = Website("https://huggingface.co")
huggingface.links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/storage',
 '/docs',
 '/enterprise',
 '/pricing',
 '/tasks',
 '/chat',
 '/collections',
 '/languages',
 '/organizations',
 '/blog',
 '/posts',
 '/papers',
 '/learn',
 '/join/discord',
 'https://discuss.huggingface.co/',
 'https://github.com/huggingface',
 '/enterprise',
 '/pro',
 '/support',
 '/inference/models',
 '/inference-endpoints',
 '/storage',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/nvidia/LocateAnything-3B',
 '/google/gemma-4-12B-it',
 '/unsloth/gemma-4-12b-it-GGUF',
 '/google/gemma-4-12B',
 '/ideogram-ai/ideogram-4-fp8',
 '/models',
 '/spaces/nvidia/LocateAnything',
 '/spaces/webml-community/bonsai-image-webgpu',
 '/spaces/VAST-AI/TripoSplat',
 '/spaces/r3gm/wan2-2-fp8da-aoti-preview-2',
 '/spaces/ideogram-ai/ideogram4',
 '/spaces',
 '/datasets/openbmb/UltraData-SFT-2605',
 '/datasets/wikimedia/structured-wikipedia',
 '/datasets/openbmb/Ultra-FineWeb-L3',
 '/datasets/ReasonCore/open-spatial-reasoning',
 '/datasets/angrygira

In [13]:
result=get_links("https://huggingface.co")
print(result)

{'links': [{'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'company page', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'github organization', 'url': 'https://github.com/huggingface'}, {'type': 'organization page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'brand page', 'url': 'https://huggingface.co/brand'}]}


## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [14]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [15]:
print(get_all_details("https://huggingface.co"))

Found links: {'links': [{'type': 'organization page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'linkedin company page', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'github organization', 'url': 'https://github.com/huggingface'}, {'type': 'twitter page', 'url': 'https://twitter.com/huggingface'}]}
Landing page:
Webpage Title:
Hugging Face – The AI community building the future.
Webpage Contents:
Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
B

In [16]:
system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."


In [17]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [18]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'company linkedin', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'github organization', 'url': 'https://github.com/huggingface'}, {'type': 'twitter', 'url': 'https://twitter.com/huggingface'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'organization page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'brand page', 'url': 'https://huggingface.co/brand'}]}


KeyboardInterrupt: 

In [19]:
def create_brochure(company_name, url):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'company page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'brand page', 'url': 'https://huggingface.co/brand'}, {'type': 'blog', 'url': 'https://huggingface.co/blog'}]}


# Discover Hugging Face  
**The AI Community Building the Future**

## Who We Are  
Hugging Face is the world's leading open platform where the machine learning (ML) community collaborates on **2M+ models**, **500k+ datasets**, and **1M+ applications (Spaces)**. We empower creators to host, share, and deploy AI across text, image, video, audio, and 3D modalities—faster and better together.

**Our Mission**: Create, discover, and collaborate on ML to accelerate innovation for everyone.

## Key Offerings  
- **Hub**: Unlimited public hosting for models, datasets, and apps. Build your ML portfolio and share with the world.  
- **Spaces**: Interactive AI apps running on Zero GPU—explore trending demos like LocateAnything, Bonsai Image, and TripoSplat.  
- **Open Source Stack**: Powering ML with libraries like **Transformers** (161k+ stars), **Diffusers**, **Datasets**, **PEFT**, and more.  
- **Enterprise Solutions**:  
  - **Team & PRO**: SSO, regions, priority support—starting at $20/user/month.  
  - **Inference Endpoints**: Deploy on GPUs from $0.60/hour.  
  - **Inference Providers**: Unified API to 45k+ models, no fees.  
  - Dedicated support, security, and private features for scale.

## Our Community & Customers  
- **50,000+ organizations** trust us, including:  
  | Organization | Type | Models Hosted |  
  |--------------|------|---------------|  
  | Google | Enterprise | 1.12k |  
  | Microsoft | Enterprise | 522 |  
  | Amazon | Enterprise | 36 |  
  | AI at Meta | Team | 2.34k |  
  | Intel | Company | 249 |  

Join our vibrant ecosystem via Discord, Forum, GitHub, and Blog.

## Careers & Culture  
We're a collaborative, open-source-first team passionate about democratizing AI. Explore exciting roles in ML engineering, research, and product at [Careers](https://huggingface.co/company/careers). Build the future with us—remote-friendly, innovative, and community-driven.

**Ready to Accelerate?** [Sign Up](https://huggingface.co/join) | [Enterprise](https://huggingface.co/enterprise) | [Explore Models](https://huggingface.co/models)  

*Hugging Face – Where ML Happens.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [20]:
def stream_brochure(company_name, url):
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        if not chunk.choices:
         continue
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [21]:
stream_brochure("SEI", "https://seic.com")

Found links: {'links': [{'type': 'about page', 'url': 'https://www.seic.com/about-sei/about-sei'}, {'type': 'company overview', 'url': 'https://www.seic.com/about-sei/overview'}, {'type': 'locations page', 'url': 'https://www.seic.com/about-sei/locations'}, {'type': 'newsroom', 'url': 'https://www.seic.com/about-sei/newsroom'}, {'type': 'contact page', 'url': 'https://www.seic.com/about-sei/contact-us'}, {'type': 'investor relations', 'url': 'https://ir.seic.com/'}, {'type': 'careers page', 'url': 'https://careers.seic.com/'}, {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/sei'}, {'type': 'Facebook', 'url': 'https://www.facebook.com/SEICorporateHeadquarters/'}, {'type': 'Instagram', 'url': 'https://www.instagram.com/sei_hq'}]}


# Discover SEI  
**Technology and Investment Solutions | Powering Financial Futures**

![SEI Logo Placeholder](https://placeholder.com/sei-logo)  
*Custom solutions. Unlimited potential. One partner.*

## Who We Are
SEI is a global leader in financial technology, operations, and investment solutions. With over 50 years of experience, we simplify the complexities of financial services, connecting every part of your operations to ignite innovation, unlock efficiency, and accelerate growth.  

We sit at the center of the financial services industry, listening, learning, and building tailored ecosystems that help clients adapt, scale, and thrive in a dynamic world. From asset managers to institutional investors, we deliver **end-to-end solutions** as your single strategic partner.

## Our Solutions for You
Select your role for customized impact:  
- **Asset Managers**: Stay ahead of change with integrated platforms.  
- **Banks & Wealth Managers**: Power the future of your business.  
- **Financial Advisors**: Simplify the complex to elevate your advice.  
- **Institutional Investors**: Investment solutions that meet your goals.  

Explore **SEI Next** for cutting-edge innovation frontiers.

## Our Customers
Trusted by leading players across the financial services ecosystem: asset managers, banks, wealth managers, financial advisors, and institutional investors worldwide. We amplify potential through holistic strategies that align vision, people, and processes.

## Company Culture: Guided by Core Values
At SEI, we defy the status quo with:  
- **Courage**: Embrace risk to drive growth.  
- **Integrity**: Act with transparency.  
- **Inclusion**: Foster respect and belonging.  
- **Collaboration**: Solve problems together.  
- **Connection**: Build lasting relationships.  
- **We Have Fun**: Celebrate creativity and community.  

We nurture inclusion via employee resource groups like **SEI Black Professionals**, philanthropy, volunteerism, and open idea exchange—uniting us for collective impact.

## Careers at SEI
Join a mission-driven team powering global finance. Explore dynamic opportunities in technology, operations, investments, and more. Thrive in an inclusive culture that values courage, collaboration, and fun.  
[Careers →](https://www.seic.com/careers)

## Investors & Partners
Half a century of growth. Access insights, news, and relations.  
[Investor Relations →](https://www.seic.com/investor-relations) | [Newsroom →](https://www.seic.com/newsroom)

**Contact Us** | **Locations** (US, EMEA, Canada) | **Client Login**  
© 2026 SEI. All rights reserved.  
[www.seic.com](https://www.seic.com) | LinkedIn | Facebook | Instagram

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>